In [0]:
# COMMAND ----------

import pandas as pd
import os

# 1. RUTA: Apuntando a la carpeta 'Datos' del repositorio clonado en tu Workspace
folder_path = "/Workspace/Users/u20216288@utp.edu.pe/Project-SQL-Caso_CancelacionesHoteleras-RADL/Data"

# 2. ESQUEMA: Destino en proyecto_kiosko
catalog_schema = "cursodataacademy.project_hotel"

# Buscamos los archivos CSV en la carpeta del repositorio
csv_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".csv")]

print(f"Se encontraron {len(csv_files)} archivos CSV:\n")

for csv_file in sorted(csv_files):
    file_path = os.path.join(folder_path, csv_file)
    table_name = csv_file.replace(".csv", "").lower()
    full_table_name = f"{catalog_schema}.{table_name}"

    # 3. MANEJO DE SEPARADORES: En la tabla cancelaciones hoteleras se usa ','
    separador = ","

    # Leer CSV con el separador dinámico
    df_pandas = pd.read_csv(file_path, encoding="utf-8-sig", sep=separador, skipinitialspace=True)

    # Reemplaza los espacios y caracteres problemáticos por guiones bajos
    df_pandas.columns = df_pandas.columns.str.replace(r'[ ,;{}()\n\t=]', '_', regex=True)

    # Crear DataFrame de Spark y guardar como tabla Delta
    df_spark = spark.createDataFrame(df_pandas)
    df_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table_name)

    row_count = df_spark.count()
    col_count = len(df_spark.columns)
    print(f"✅ {csv_file} → {full_table_name} ({row_count} filas, {col_count} columnas: {df_spark.columns})")

print(f"\n✨ Carga completa: {len(csv_files)} tablas creadas en {catalog_schema}")

In [0]:
%sql
--Creacion de la tabla tb_reservas
CREATE OR REPLACE TABLE cursodataacademy.project_hotel.tb_reservas AS
SELECT 
    hotel,
    is_canceled,
    lead_time,
    arrival_date_year,
    arrival_date_month,
    arrival_date_day_of_month,
    distribution_channel,
    reservation_status_date,
    adr,
    children,
    babies,
    total_of_special_requests,
    booking_changes,
    country,
    agent
FROM cursodataacademy.project_hotel.hotel_bookings;

In [0]:
%sql --Creacion de la tabla tb_reservas_prueba
CREATE OR REPLACE TABLE cursodataacademy.project_hotel.tb_reservas_prueba AS
SELECT * 
FROM cursodataacademy.project_hotel.tb_reservas

In [0]:
%sql 
--Crear la columna arrival_Date para ejercicio 9
ALTER TABLE cursodataacademy.project_hotel.tb_reservas
ADD COLUMN arrival_date DATE;


In [0]:
%sql 
--Actualizar la columna arrival_date como fecha
UPDATE cursodataacademy.project_hotel.tb_reservas
SET arrival_date = TO_DATE(
                            CONCAT_WS('-',
                              CAST(arrival_date_day_of_month AS STRING),
                              CAST(arrival_date_month AS STRING),
                              CAST(arrival_date_year AS STRING)),
                            'd-MMMM-yyyy');

In [0]:
%sql
-- Tabla de países 
CREATE OR REPLACE TABLE cursodataacademy.project_hotel.tb_paises (
  cod_pais    STRING,
  nombre_pais STRING 
);

In [0]:
%sql
INSERT INTO cursodataacademy.project_hotel.tb_paises (cod_pais, nombre_pais) VALUES
  ('ABW', 'Aruba'),
  ('AGO', 'Angola'),
  ('AIA', 'Anguila'),
  ('ALB', 'Albania'),
  ('AND', 'Andorra'),
  ('ARE', 'Emiratos Árabes Unidos'),
  ('ARG', 'Argentina'),
  ('ARM', 'Armenia'),
  ('ASM', 'Samoa Americana'),
  ('ATA', 'Antártida'),
  ('ATF', 'Tierras Australes y Antárticas Francesas'),
  ('AUS', 'Australia'),
  ('AUT', 'Austria'),
  ('AZE', 'Azerbaiyán'),
  ('BDI', 'Burundi'),
  ('BEL', 'Bélgica'),
  ('BEN', 'Benín'),
  ('BFA', 'Burkina Faso'),
  ('BGD', 'Bangladés'),
  ('BGR', 'Bulgaria'),
  ('BHR', 'Baréin'),
  ('BHS', 'Bahamas'),
  ('BIH', 'Bosnia y Herzegovina'),
  ('BLR', 'Bielorrusia'),
  ('BOL', 'Bolivia'),
  ('BRA', 'Brasil'),
  ('BRB', 'Barbados'),
  ('BWA', 'Botsuana'),
  ('CAF', 'República Centroafricana'),
  ('CHE', 'Suiza'),
  ('CHL', 'Chile'),
  ('CHN', 'China'),
  ('CIV', 'Costa de Marfil'),
  ('CMR', 'Camerún'),
  ('CN', 'China (código no estándar)'),
  ('COL', 'Colombia'),
  ('COM', 'Comoras'),
  ('CPV', 'Cabo Verde'),
  ('CRI', 'Costa Rica'),
  ('CUB', 'Cuba'),
  ('CYM', 'Islas Caimán'),
  ('CYP', 'Chipre'),
  ('CZE', 'Chequia'),
  ('DEU', 'Alemania'),
  ('DJI', 'Yibuti'),
  ('DMA', 'Dominica'),
  ('DNK', 'Dinamarca'),
  ('DOM', 'República Dominicana'),
  ('DZA', 'Argelia'),
  ('ECU', 'Ecuador'),
  ('EGY', 'Egipto'),
  ('ESP', 'España'),
  ('EST', 'Estonia'),
  ('ETH', 'Etiopía'),
  ('FIN', 'Finlandia'),
  ('FJI', 'Fiyi'),
  ('FRA', 'Francia'),
  ('FRO', 'Islas Feroe'),
  ('GAB', 'Gabón'),
  ('GBR', 'Reino Unido'),
  ('GEO', 'Georgia'),
  ('GGY', 'Guernsey'),
  ('GHA', 'Ghana'),
  ('GIB', 'Gibraltar'),
  ('GLP', 'Guadalupe'),
  ('GNB', 'Guinea-Bisáu'),
  ('GRC', 'Grecia'),
  ('GTM', 'Guatemala'),
  ('GUY', 'Guyana'),
  ('HKG', 'Hong Kong'),
  ('HND', 'Honduras'),
  ('HRV', 'Croacia'),
  ('HUN', 'Hungría'),
  ('IDN', 'Indonesia'),
  ('IMN', 'Isla de Man'),
  ('IND', 'India'),
  ('IRL', 'Irlanda'),
  ('IRN', 'Irán'),
  ('IRQ', 'Irak'),
  ('ISL', 'Islandia'),
  ('ISR', 'Israel'),
  ('ITA', 'Italia'),
  ('JAM', 'Jamaica'),
  ('JEY', 'Jersey'),
  ('JOR', 'Jordania'),
  ('JPN', 'Japón'),
  ('KAZ', 'Kazajistán'),
  ('KEN', 'Kenia'),
  ('KHM', 'Camboya'),
  ('KIR', 'Kiribati'),
  ('KNA', 'San Cristóbal y Nieves'),
  ('KOR', 'Corea del Sur'),
  ('KWT', 'Kuwait'),
  ('LAO', 'Laos'),
  ('LBN', 'Líbano'),
  ('LBY', 'Libia'),
  ('LCA', 'Santa Lucía'),
  ('LIE', 'Liechtenstein'),
  ('LKA', 'Sri Lanka'),
  ('LTU', 'Lituania'),
  ('LUX', 'Luxemburgo'),
  ('LVA', 'Letonia'),
  ('MAC', 'Macao'),
  ('MAR', 'Marruecos'),
  ('MCO', 'Mónaco'),
  ('MDG', 'Madagascar'),
  ('MDV', 'Maldivas'),
  ('MEX', 'México'),
  ('MKD', 'Macedonia del Norte'),
  ('MLI', 'Malí'),
  ('MLT', 'Malta'),
  ('MMR', 'Birmania (Myanmar)'),
  ('MNE', 'Montenegro'),
  ('MOZ', 'Mozambique'),
  ('MRT', 'Mauritania'),
  ('MUS', 'Mauricio'),
  ('MWI', 'Malaui'),
  ('MYS', 'Malasia'),
  ('MYT', 'Mayotte'),
  ('NAM', 'Namibia'),
  ('NCL', 'Nueva Caledonia'),
  ('NGA', 'Nigeria'),
  ('NIC', 'Nicaragua'),
  ('NLD', 'Países Bajos'),
  ('NOR', 'Noruega'),
  ('NPL', 'Nepal'),
  ('NZL', 'Nueva Zelanda'),
  ('OMN', 'Omán'),
  ('PAK', 'Pakistán'),
  ('PAN', 'Panamá'),
  ('PER', 'Perú'),
  ('PHL', 'Filipinas'),
  ('PLW', 'Palaos'),
  ('POL', 'Polonia'),
  ('PRI', 'Puerto Rico'),
  ('PRT', 'Portugal'),
  ('PRY', 'Paraguay'),
  ('PYF', 'Polinesia Francesa'),
  ('QAT', 'Catar'),
  ('ROU', 'Rumanía'),
  ('RUS', 'Rusia'),
  ('RWA', 'Ruanda'),
  ('SAU', 'Arabia Saudita'),
  ('SDN', 'Sudán'),
  ('SEN', 'Senegal'),
  ('SGP', 'Singapur'),
  ('SLE', 'Sierra Leona'),
  ('SLV', 'El Salvador'),
  ('SMR', 'San Marino'),
  ('SRB', 'Serbia'),
  ('STP', 'Santo Tomé y Príncipe'),
  ('SUR', 'Surinam'),
  ('SVK', 'Eslovaquia'),
  ('SVN', 'Eslovenia'),
  ('SWE', 'Suecia'),
  ('SYC', 'Seychelles'),
  ('SYR', 'Siria'),
  ('TGO', 'Togo'),
  ('THA', 'Tailandia'),
  ('TJK', 'Tayikistán'),
  ('TMP', 'Timor Oriental (código antiguo)'),
  ('TUN', 'Túnez'),
  ('TUR', 'Turquía'),
  ('TWN', 'Taiwán'),
  ('TZA', 'Tanzania'),
  ('UGA', 'Uganda'),
  ('UKR', 'Ucrania'),
  ('UMI', 'Islas Menores Alejadas de EE. UU.'),
  ('URY', 'Uruguay'),
  ('USA', 'Estados Unidos'),
  ('UZB', 'Uzbekistán'),
  ('VEN', 'Venezuela'),
  ('VGB', 'Islas Vírgenes Británicas'),
  ('VNM', 'Vietnam'),
  ('ZAF', 'Sudáfrica'),
  ('ZMB', 'Zambia'),
  ('ZWE', 'Zimbabue');

In [0]:
%sql
CREATE OR REPLACE TABLE cursodataacademy.project_hotel.tb_vendedores AS
SELECT DISTINCT
  CAST(agent AS INT)                                       AS id_vendedor,
  CONCAT('Agente ', LPAD(CAST(id_vendedor as string), 3, '0'))   AS nombre_vendedor
FROM cursodataacademy.project_hotel.tb_reservas
WHERE agent IS NOT NULL;